### Task 3: RAG Core Logic and Evaluation

#### Setup & Imports

In [1]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq
from tqdm import tqdm

load_dotenv()


api_key = os.getenv("GROQ_API_KEY")
if api_key:
    print("Groq API key loaded successfully")
else:
    print("ERROR: GROQ_API_KEY not found")

print("All libraries imported successfully!")

Groq API key loaded successfully
All libraries imported successfully!


#### Load pre-built vector store

In [2]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))

PARQUET_PATH = os.path.join(BASE_DIR, "data", "raw", "complaint_embeddings.parquet")

print("Loading pre-built embeddings")
df_embeddings = pd.read_parquet(PARQUET_PATH)

print(f"Loaded successfully!")
print(f"Shape: {df_embeddings.shape}")
print(f"Columns: {df_embeddings.columns.tolist()}")
print(f"\nFirst row preview:")
print(df_embeddings.head(1))

Loading pre-built embeddings


Loaded successfully!
Shape: (1375327, 4)
Columns: ['id', 'document', 'embedding', 'metadata']

First row preview:
           id                                           document  \
0  14069121_0  a card was opened under my name by a fraudster...   

                                           embedding  \
0  [-0.04277738183736801, 0.025624370202422142, -...   

                                            metadata  
0  {'chunk_index': 0, 'company': 'CITIBANK, N.A.'...  


#### Inspect metadata

In [3]:
import ast

first_metadata = df_embeddings['metadata'].iloc[0]

print("Metadata fields available:")
print(first_metadata)
print(f"\nType: {type(first_metadata)}")

print("\nMetadata from row 100:")
print(df_embeddings['metadata'].iloc[100])

Metadata fields available:
{'chunk_index': 0, 'company': 'CITIBANK, N.A.', 'complaint_id': '14069121', 'date_received': '2025-06-13', 'issue': 'Getting a credit card', 'product': 'Credit card', 'product_category': 'Credit Card', 'state': 'TX', 'sub_issue': 'Card opened without my consent or knowledge', 'total_chunks': 1}

Type: <class 'dict'>

Metadata from row 100:
{'chunk_index': 2, 'company': 'JPMORGAN CHASE & CO.', 'complaint_id': '12511020', 'date_received': '2025-03-17', 'issue': 'Problem with a purchase shown on your statement', 'product': 'Credit card', 'product_category': 'Credit Card', 'state': 'NJ', 'sub_issue': "Credit card company isn't resolving a dispute about a purchase on your statement", 'total_chunks': 3}


### Load ChromaDB

In [4]:
VECTOR_STORE_PATH = os.path.join(BASE_DIR, "vector_store", "prebuilt_store")

client = chromadb.PersistentClient(path=VECTOR_STORE_PATH)

existing = [c.name for c in client.list_collections()]

if "complaints_full" in existing:
    collection = client.get_collection("complaints_full")
    print(f"Total chunks in store: {collection.count():,}")
else:
    print("Building ChromaDB from pre-built embeddings")
    
    collection = client.create_collection(
        name="complaints_full",
        metadata={"hnsw:space": "cosine"}
    )
    
    
    BATCH_SIZE = 2000
    total = len(df_embeddings)
    num_batches = (total // BATCH_SIZE) + 1
    
    for batch_num in tqdm(range(num_batches), desc="Indexing chunks"):
        start = batch_num * BATCH_SIZE
        end = min(start + BATCH_SIZE, total)
        
        if start >= total:
            break
            
        batch = df_embeddings.iloc[start:end]
        
        
        batch_embeddings = np.array(batch['embedding'].tolist())
        
       
        batch_metadata = []
        for meta in batch['metadata']:
            clean_meta = {k: str(v) if v is not None else "" 
                         for k, v in meta.items()}
            batch_metadata.append(clean_meta)
        
        collection.add(
            ids=batch['id'].tolist(),
            embeddings=batch_embeddings.tolist(),
            documents=batch['document'].tolist(),
            metadatas=batch_metadata
        )
    
    print(f"\nDone! Total chunks indexed: {collection.count():,}")
    print(f"Saved to: {VECTOR_STORE_PATH}")

Total chunks in store: 1,375,327


### Retriver

In [5]:
import sys
sys.path.insert(0, BASE_DIR)

from src.retriever import retrieve, format_context

question = "Why are customers unhappy with credit cards?"

print(f"Question: '{question}'")
print("Searching 1.37 million chunks...\n")

chunks = retrieve(
    question=question,
    collection=collection,
    n_results=5
)

print(f"Retrieved {len(chunks)} chunks\n")


for i, chunk in enumerate(chunks, 1):
    print(f"\nResult {i}:")
    print(f"  Product:    {chunk['metadata']['product_category']}")
    print(f"  Issue:      {chunk['metadata']['issue']}")
    print(f"  Company:    {chunk['metadata']['company']}")
    print(f"  Similarity: {chunk['similarity_score']}")
    print(f"  Text:       {chunk['text'][:200]}...")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!
Question: 'Why are customers unhappy with credit cards?'
Searching 1.37 million chunks...

Retrieved 5 chunks


Result 1:
  Product:    Credit Card
  Issue:      Other features, terms, or problems
  Company:    BARCLAYS BANK DELAWARE
  Similarity: 0.7136
  Text:       card company and was very unhappy and frustrated. as a consumer i feel that we apply for new credit cards because of the features and benefits they offer, however we need to understand how to use them...

Result 2:
  Product:    Credit Card
  Issue:      Other features, terms, or problems
  Company:    DISCOVER BANK
  Similarity: 0.7113
  Text:       creditors. i have an exceptional payment history. there was no reason for them to reduce my credit limit at all doing so caused me harm by making my credit usage shoot up. what the is up with these co...

Result 3:
  Product:    Credit Card
  Issue:      Fees or interest
  Company:    AMERICAN EXPRESS COMPANY
  Similarity: 0.7105
  Text:       credit c

### Prompt engineering

In [6]:
from src.retriever import format_context
from src.generator import build_prompt

context = format_context(chunks)

print("FORMATTED CONTEXT:")
print(context[:1000])
print("\n... (truncated for display)")

prompt = build_prompt(question, context)

print("\n\nCOMPLETE PROMPT SENT TO LLM:")
print(prompt[:500])
print("\n... (truncated for display)")

FORMATTED CONTEXT:
[Complaint 1]
Product: Credit Card
Issue: Other features, terms, or problems
Company: BARCLAYS BANK DELAWARE
Text: card company and was very unhappy and frustrated. as a consumer i feel that we apply for new credit cards because of the features and benefits they offer, however we need to understand how to use them. i am not happy with the customer service and i am not happy with the misinformation i was given. i have been given misinformation by several customer service representatives and i feel that the credit card company is taking advantage of consumers who dont understand the features and benefits.

---
[Complaint 2]
Product: Credit Card
Issue: Other features, terms, or problems
Company: DISCOVER BANK
Text: creditors. i have an exceptional payment history. there was no reason for them to reduce my credit limit at all doing so caused me harm by making my credit usage shoot up. what the is up with these companies it like here is the card but don't use it!

---
[Co

### Generator

In [7]:
import os
from groq import Groq
from dotenv import load_dotenv
from src.retriever import format_context
from src.generator import build_prompt

load_dotenv()


GROQ_MODEL = "llama-3.1-8b-instant"

prompt = build_prompt(question, context)

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

print("Sending to Groq LLM...")
print(f"Model: {GROQ_MODEL}\n")

response = client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful financial analyst. Answer based only on provided complaint data."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    max_tokens=1024,
    temperature=0.1
)

answer = response.choices[0].message.content.strip()

print("ANSWER:")
print(answer)
print(f"Model: {GROQ_MODEL}")
print(f"Tokens used: {response.usage.prompt_tokens} prompt + {response.usage.completion_tokens} completion")

Sending to Groq LLM...
Model: llama-3.1-8b-instant

ANSWER:
Based on the complaint excerpts provided, customers are unhappy with credit cards due to the following reasons:

1. Misinformation and poor customer service (Complaint 1: BARCLAYS BANK DELAWARE)
2. Unexpected changes to credit limits (Complaint 2: DISCOVER BANK)
3. Indifferent response to long-term customers and unexpected interest rate increases (Complaint 3: AMERICAN EXPRESS COMPANY)
4. Problematic customer service (Complaint 4: JPMORGAN CHASE & CO.)
5. Thoughtless and unhelpful marketing and promotional actions, particularly during times of economic stress (Complaint 5: CITIBANK, N.A.)

These issues are related to the credit card products offered by the following companies: BARCLAYS BANK DELAWARE, DISCOVER BANK, AMERICAN EXPRESS COMPANY, JPMORGAN CHASE & CO., and CITIBANK, N.A.
Model: llama-3.1-8b-instant
Tokens used: 688 prompt + 186 completion


### Full RAG pipeline

In [8]:
from src.rag_pipeline import run_rag_pipeline, display_result

result = run_rag_pipeline(
    question="Why are customers unhappy with credit cards?",
    collection=collection,
    n_results=5
)

display_result(result)

Searching complaints for: 'Why are customers unhappy with credit cards?'
Retrieved 5 relevant chunks
Generating answer...
QUESTION: Why are customers unhappy with credit cards?

ANSWER:
Based on the complaint excerpts provided, customers are unhappy with credit cards due to the following reasons:

1. Misinformation and poor customer service (Complaint 1: BARCLAYS BANK DELAWARE)
2. Unexpected changes to credit limits (Complaint 2: DISCOVER BANK)
3. Indifferent response to long-term customers and unexpected interest rate increases (Complaint 3: AMERICAN EXPRESS COMPANY)
4. Problematic customer service (Complaint 4: JPMORGAN CHASE & CO.)
5. Thoughtless and unhelpful marketing actions, particularly during times of economic stress (Complaint 5: CITIBANK, N.A.)

These complaints highlight various issues with credit card products and services offered by multiple companies, including BARCLAYS BANK DELAWARE, DISCOVER BANK, AMERICAN EXPRESS COMPANY, JPMORGAN CHASE & CO., and CITIBANK, N.A.
SOURC

### Evaluation

Running 8 representative questions through the RAG pipeline.
Score each answer on a scale of 1-5 for quality.

Scoring guide:
- 5 = Perfect. Accurate, relevant, grounded in evidence
- 4 = Good. Mostly accurate with minor gaps  
- 3 = Acceptable. Relevant but missing key points
- 2 = Poor. Partially relevant or incomplete
- 1 = Bad. Wrong, irrelevant, or hallucinated

In [9]:
eval_questions = [
    {"question": "Why are customers unhappy with credit cards?", 
     "product": "Credit Card"},
    
    {"question": "What are the most common issues with money transfers?", 
     "product": "Money Transfer"},
    
    {"question": "Why do customers complain about savings accounts?", 
     "product": "Savings Account"},
    
    {"question": "What problems do customers face with personal loans?", 
     "product": "Personal Loan"},
    
    {"question": "What fraud issues do customers report with credit cards?", 
     "product": "Credit Card"},
    
    {"question": "Why are customers disputing charges on their accounts?", 
     "product": None},
    
    {"question": "What are common problems with account access and login?", 
     "product": "Savings Account"},
    
    {"question": "Which companies receive the most complaints about fees?", 
     "product": None},
]

eval_results = []

print("Running evaluation...\n")

for i, item in enumerate(eval_questions, 1):
    print(f"\nQuestion {i}/{len(eval_questions)}: {item['question']}")
    
    result = run_rag_pipeline(
        question=item['question'],
        collection=collection,
        n_results=5,
        product_filter=item['product']
    )
    
    eval_results.append({
        "question": item['question'],
        "product_filter": item['product'],
        "answer": result['answer'],
        "sources": result['retrieved_chunks'][:2],
        "tokens": result['tokens_used']
    })
    
    print(f"Answer preview: {result['answer'][:150]}...")

print(f"\nAll {len(eval_questions)} questions evaluated!")

Running evaluation...


Question 1/8: Why are customers unhappy with credit cards?
Searching complaints for: 'Why are customers unhappy with credit cards?'
Retrieved 5 relevant chunks
Generating answer...
Answer preview: Based on the complaint excerpts provided, customers are unhappy with credit cards due to the following reasons:

1. Misinformation and poor customer s...

Question 2/8: What are the most common issues with money transfers?
Searching complaints for: 'What are the most common issues with money transfers?'
Retrieved 5 relevant chunks
Generating answer...
Answer preview: Based on the complaint excerpts provided, the most common issues with money transfers are:

1. **Delays or unavailability of funds**: Complaints 2, 4,...

Question 3/8: Why do customers complain about savings accounts?
Searching complaints for: 'Why do customers complain about savings accounts?'
Retrieved 5 relevant chunks
Generating answer...
Answer preview: Based on the complaint excerpts provided, custo

### Evaluation Table

In [12]:
for i, result in enumerate(eval_results, 1):
    print(f"Q{i}: {result['question']}")
    print(result['answer'])
    print(f"\nSource 1: {result['sources'][0]['metadata']['company']} | "
          f"{result['sources'][0]['metadata']['issue']}")
    print(f"Source 2: {result['sources'][1]['metadata']['company']} | "
          f"{result['sources'][1]['metadata']['issue']}")
    print(f"Tokens: {result['tokens']}")
    print()

Q1: Why are customers unhappy with credit cards?
Based on the complaint excerpts provided, customers are unhappy with credit cards due to the following reasons:

1. Misinformation and poor customer service (Complaint 1: BARCLAYS BANK DELAWARE)
2. Unexpected changes to credit limits (Complaint 2: DISCOVER BANK)
3. Indifferent response to long-term customers and unexpected interest rate increases (Complaint 3: AMERICAN EXPRESS COMPANY)
4. Problematic customer service (Complaint 4: JPMORGAN CHASE & CO.)
5. Thoughtless and unhelpful marketing and promotional actions, particularly during times of economic stress (Complaint 5: CITIBANK, N.A.)

These complaints highlight various issues with credit card products and services, including customer service, communication, and marketing practices.

Source 1: BARCLAYS BANK DELAWARE | Other features, terms, or problems
Source 2: DISCOVER BANK | Other features, terms, or problems
Tokens: 847

Q2: What are the most common issues with money transfers?
B